# Create and read a policy

`POST /admin/policies` stands in for HR staff adding a policy. The connector
polls `updated_since` (every 10 seconds) and writes `raw/hr/policy/{id}.md`
with version, content hash, ACL, source time, and snapshot time.

Requires `make up`. The cleanup cell deletes the row and waits for the
reconcile sweep, which is about a minute.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [2]:
POLICY_ID = "pol-demo-create"
KEY = f"raw/hr/policy/{POLICY_ID}.md"
BODY = "# Demo policy\n\nCreated by the notebook.\n"
PAYLOAD = {
    "policy_id": POLICY_ID,
    "title": "Demo policy",
    "category": "leave",
    "region": "DE",
    "format": "markdown",
    "native_acl": ["hr_employee"],
    "body": BODY,
}
hr_demo.delete_if_present("policy", POLICY_ID)
created = hr_demo.admin("POST", "/admin/policies", PAYLOAD)
print(created["policy_id"], "version", created["version"], created["native_acl"])


pol-demo-create version 1 ['hr_employee']


## Snapshot


In [3]:
def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        BODY.encode() in hr_demo.object_body(KEY)
        and meta.get(hr_demo.META_ACL) == "hr_employee"
        and meta.get(hr_demo.META_VERSION) == str(created["version"])
        and len(meta.get(hr_demo.META_HASH, "")) == 64
        and all(meta.get(name) for name in hr_demo.META_KEYS)
    )

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "create snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


{'kbs-content-hash': '4166869738e975d3f2747dac7ca2a132bd0345f3e6a30c2314bdde19821d5b9d', 'kbs-native-acl': 'hr_employee', 'kbs-snapshot-at': '2026-10-12T08:46:18Z', 'kbs-source-updated-at': '2026-10-12T08:46:14.886511Z', 'kbs-version': '1'}


## Negative cases

A second create of the same id is a conflict. An id that fails the pattern
is rejected before it is stored.


In [4]:
duplicate = hr_demo.expect_status("POST", "/admin/policies", 409, PAYLOAD)
print("duplicate", duplicate.status_code)
bad = {**PAYLOAD, "policy_id": "pol-NO"}
rejected = hr_demo.expect_status("POST", "/admin/policies", 422, bad)
print("bad id", rejected.status_code)


duplicate 409
bad id 422


## Cleanup


In [5]:
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)


EndpointConnectionError: Could not connect to the endpoint URL: "http://floci:4566/kbs-raw-snapshots?versions&prefix=raw%2Fhr%2Fpolicy%2Fpol-demo-create.md&encoding-type=url"